# (MJF-1B): Parkinson's Freezing of Gait

In [ ]:
# Import Python libraries
import numpy as np         # linear algebra
import pandas as pd        # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns   
from scipy.signal import butter, filtfilt
from scipy.stats import skew, kurtosis
import tsfresh     
import os
from sklearn.preprocessing import StandardScaler

import polars as pl
import dask.dataframe as dd
from pathlib import Path

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn import svm
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import scale
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve
from sklearn.metrics import accuracy_score, f1_score, precision_score, roc_auc_score, roc_curve, auc, confusion_matrix, classification_report

# Data Exploration

In [ ]:
# File paths for three training datasets
defog = Path('/kaggle/input/tlvmc-parkinsons-freezing-gait-prediction/train/defog')
notype = Path('/kaggle/input/tlvmc-parkinsons-freezing-gait-prediction/train/notype')
tdcsfog = Path('/kaggle/input/tlvmc-parkinsons-freezing-gait-prediction/train/tdcsfog')

### Add patientID column

In [ ]:
# Read defog dataset and add patientID 
defog_files = [f for f in os.listdir(defog) if f.endswith('.csv')]
defog_list = []

for path in defog.glob("*.csv"):
    patient_id = path.stem  # removes .csv
    df = pl.read_csv(path)
    df = df.with_columns([
        pl.lit(patient_id).alias("patient_id")])
    defog_list.append(df)

defog_df = pl.concat(defog_list)

In [ ]:
# Read tdcsfog dataset and add patientID 
tdcsfog_files = [f for f in os.listdir(tdcsfog) if f.endswith('.csv')]
tdcsfog_list = []

for path in tdcsfog.glob("*.csv"):
    patient_id = path.stem  # removes .csv
    df = pl.read_csv(path)
    df = df.with_columns([
        pl.lit(patient_id).alias("patient_id")])
    tdcsfog_list.append(df)

tdcsfog_df = pl.concat(tdcsfog_list)

### Explore defog

In [ ]:
defog_df.shape

In [ ]:
defog_df.head()

In [ ]:
print(f'DEFOG COLUMNS:\n{defog_df.columns}')

In [ ]:
print(f'DEFOG DATA TYPES:\n{defog_df.dtypes}')

### Explore tdcsfog

In [ ]:
tdcsfog_df.shape

In [ ]:
tdcsfog_df.head()

In [ ]:
print(f'TDCSFOG COLUMNS:\n{tdcsfog_df.columns}')

In [ ]:
print(f'TDCSFOG DATA TYPES:\n{tdcsfog_df.dtypes}')

# Visualize Acceleration w/ FoG Events BEFORE Cleaning

In [ ]:
# Get patient IDs with FoG events
patient_id_defog = 'be9d33541d'
patient_id_tdcsfog = 'b670d1cddd' 

def _spans_from_mask(x, mask):
    """Convert a boolean mask to contiguous (start, end) intervals."""
    mask = np.asarray(mask, dtype=bool)
    if mask.size == 0:
        return []
    change = np.diff(mask.astype(int), prepend=0)
    starts = np.where(change == 1)[0]
    ends   = np.where(change == -1)[0] - 1
    if mask[0]:  starts = np.r_[0, starts]
    if mask[-1]: ends   = np.r_[ends, len(mask)-1]
    return [(x[s], x[e]) for s, e in zip(starts, ends)]

def plot_patient(ax, pl_df, pid, title):
    df = pl_df.filter(pl.col("patient_id") == pid).to_pandas()

    # Plot accelerations 
    for col in ["AccV", "AccML", "AccAP"]:
        if col in df.columns:
            ax.plot(df["Time"], df[col], label=col, alpha=0.8)

    if "StartHesitation" in df.columns:
        sh = pd.to_numeric(df["StartHesitation"], errors="coerce").fillna(0).to_numpy()
        mask = sh > 0
        spans = _spans_from_mask(df["Time"].to_numpy(), mask)

        shaded_once = False
        for (start, end) in spans:
            ax.axvspan(start, end, alpha=0.25, label=(None if shaded_once else "StartHesitation"))
            shaded_once = True

    if "Turn" in df.columns:
        sh = pd.to_numeric(df["Turn"], errors="coerce").fillna(0).to_numpy()
        mask = sh > 0
        spans = _spans_from_mask(df["Time"].to_numpy(), mask)

        shaded_once = False
        onset_once = False
        for (start, end) in spans:
            ax.axvspan(start, end, color="green", alpha=0.25, label=(None if shaded_once else "Turn"))
            shaded_once = True

    if "Walking" in df.columns:
        sh = pd.to_numeric(df["Walking"], errors="coerce").fillna(0).to_numpy()
        mask = sh > 0
        spans = _spans_from_mask(df["Time"].to_numpy(), mask)

        shaded_once = False
        for (start, end) in spans:
            ax.axvspan(start, end, color="orange", alpha=0.25, label=(None if shaded_once else "Walking"))
            shaded_once = True
            
    ax.set_title(title)
    ax.set_xlabel("Time")
    ax.set_ylabel("Acceleration (g)")
    ax.grid(True)

# Create stacked subplots
fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(15, 10), sharex=False)

plot_patient(axes[0], defog_df,   patient_id_defog,   f"DEFoG — {patient_id_defog}")
plot_patient(axes[1], tdcsfog_df, patient_id_tdcsfog, f"TDCSFoG — {patient_id_tdcsfog}")

# Clean legends 
for ax in axes:
    h, l = ax.get_legend_handles_labels()
    seen, H, L = set(), [], []
    for hi, li in zip(h, l):
        if li and li not in seen:
            H.append(hi); L.append(li); seen.add(li)
    ax.legend(H, L, loc="upper right", ncol=3, fontsize=8)

fig.suptitle("Acceleration with FoG Highlights (DEFoG & tDCSFoG)", y=0.98)
plt.tight_layout()
plt.show()

# Data Cleaning

In [ ]:
# Data types of features 
print(f'DEFOG DATA TYPES:\n{defog_df.dtypes}\n')
print(f'TDCSFOG DATA TYPES:\n{tdcsfog_df.dtypes}\n')

In [ ]:
# Count nulls
tdcsfog_df.null_count()

In [ ]:
# Convert acceleration units in defog to m/s^2
G_CONVERSION = 9.80665
defog_df[["AccV", "AccML", "AccAP"]] *= G_CONVERSION

In [ ]:
# Convert the Valid and Task Columns in defog to Integer Columns
def convert_valid_and_t(df):
    df = df.with_columns(
        pl.col("Valid").cast(pl.Int8).alias("Valid"))
    
    df = df.with_columns(
        pl.col("Task").cast(pl.Int8).alias("Task"))
    
    return df
    
defog_df = convert_valid_and_t(defog_df)

In [ ]:
# Create acceleration magnitude column
def acc_magnitude(df):
    df = df.with_columns(
        (
            (pl.col("AccV") ** 2 + pl.col("AccML") ** 2 + pl.col("AccAP") ** 2).sqrt()
        ).alias("Acc_MAGNITUDE")
    )

    return df

tdcsfog_df = acc_magnitude(tdcsfog_df)
defog_df = acc_magnitude(defog_df)

In [ ]:
# Standardize acceleration per patient for each training dataframe
def standardize_acc_by_patient(df: pl.DataFrame):
    acc_cols = ['AccV', 'AccML', 'AccAP']
    for col in acc_cols:
        df = df.with_columns(
            (
                (pl.col(col) - pl.col(col).mean().over("patient_id")) /
                pl.col(col).std().over("patient_id")
            ).alias(col) 
        )
    return df

tdcsfog_df = standardize_acc_by_patient(tdcsfog_df)
defog_df = standardize_acc_by_patient(defog_df)

In [ ]:
# Create a new column where Time is in seconds and starts at 0 for patient
def time_to_seconds(df, hertz):
    df = df.with_columns([
        ((pl.col("Time") / hertz) - (pl.col("Time") / hertz).min().over("patient_id"))
        .alias("Time_in_sec")])
    return df

tdcsfog_df = time_to_seconds(tdcsfog_df, 128)
defog_df   = time_to_seconds(defog_df, 100)

In [ ]:
defog_df.head()

### Band-pass Filter

In [ ]:
# Band-pass Filter 
def infer_fs(time_seconds: np.ndarray) -> float:
    dt = np.diff(np.asarray(time_seconds, dtype=float))
    dt = dt[np.isfinite(dt) & (dt > 0)]
    return 1.0 / np.median(dt)

def design_bandpass(low_hz: float, high_hz: float, fs: float, order: int = 4):
    nyq = fs / 2.0
    low = max(1e-6, low_hz / nyq)
    high = min(0.999999, high_hz / nyq)
   
    b, a = butter(order, [low, high], btype="band")
    return b, a

def bandpass_series(y: pd.Series, b, a) -> np.ndarray:
    sig = pd.to_numeric(y, errors="coerce").interpolate(limit_direction="both").to_numpy(float)
    return filtfilt(b, a, sig, method="pad")

def bandpass_dataframe(df: pd.DataFrame, cols=('AccV','AccML','AccAP'),
                       low_hz=0.1, high_hz=30.0, order=4) -> pd.DataFrame:
    out = df.copy()
    # Only keep columns that exist
    cols = tuple([c for c in cols if c in out.columns])
    if len(cols) == 0:
        return out

    fs = infer_fs(out['Time_in_sec'].to_numpy())
    b, a = design_bandpass(low_hz, high_hz, fs, order)
    for col in cols:
        out[f"{col}_bp"] = bandpass_series(out[col], b, a)
    return out

In [ ]:
# Apply Band-pass to all patients 
def add_bandpass_to_all_patients(pl_df: pl.DataFrame,
                                 cols=('AccV','AccML','AccAP'),
                                 low_hz=0.1, high_hz=30.0, order=4) -> pl.DataFrame:

    out_chunks = []
    # Unique patient list
    patient_ids = pl_df.select("patient_id").unique().to_series().to_list()

    for pid in patient_ids:
        g = pl_df.filter(pl.col("patient_id") == pid).to_pandas()
        # Skip tiny or malformed groups
        if "Time_in_sec" not in g.columns or len(g) < 5:
            out_chunks.append(pl.from_pandas(g))  # just pass through
            continue
        try:
            g_bp = bandpass_dataframe(g, cols=cols, low_hz=low_hz, high_hz=high_hz, order=order)
        except Exception as e:
            print(f"[WARN] Skipping bandpass for patient {pid}: {e}")
            g_bp = g  # pass through raw if something fails

        out_chunks.append(pl.from_pandas(g_bp))

    return pl.concat(out_chunks, how="vertical_relaxed")

defog_df_bp   = add_bandpass_to_all_patients(defog_df,   cols=('AccV','AccML','AccAP'),
                                             low_hz=0.1, high_hz=30.0, order=4)
tdcsfog_df_bp = add_bandpass_to_all_patients(tdcsfog_df, cols=('AccV','AccML','AccAP'),
                                             low_hz=0.1, high_hz=30.0, order=4)

print("DEFOG with band-pass columns:", [c for c in defog_df_bp.columns if c.endswith("_bp")][:6], "...")
print("TDCSFOG with band-pass columns:", [c for c in tdcsfog_df_bp.columns if c.endswith("_bp")][:6], "...")

# Visualize Data AFTER Cleaning

### Raw v.s. BP Acceleration Magnitude 

In [ ]:
def add_magnitude_cols(pl_df: pl.DataFrame) -> pl.DataFrame: 
    out = pl_df.with_columns(
        ((pl.col("AccV")**2 + pl.col("AccML")**2 + pl.col("AccAP")**2).sqrt()).alias("AccMag")
    )
    bp_cols = {"AccV_bp", "AccML_bp", "AccAP_bp"}
    if bp_cols.issubset(set(out.columns)):
        out = out.with_columns(
            ((pl.col("AccV_bp")**2 + pl.col("AccML_bp")**2 + pl.col("AccAP_bp")**2).sqrt()).alias("AccMag_bp")
        )
    return out 
    
def plot_patient_mag(pl_df: pl.DataFrame, patient_id: str,
                     time_col: str = "Time_in_sec",
                     show_events: bool = True,
                     title_suffix: str = ""):
    dfp = pl_df.filter(pl.col("patient_id") == patient_id).to_pandas()

    plt.figure(figsize=(16,6))
    plt.plot(dfp[time_col], dfp["AccMag"], label="|a| (raw)", alpha=0.7)
    if "AccMag_bp" in dfp.columns:
        plt.plot(dfp[time_col], dfp["AccMag_bp"], label="|a| (0.1–30 Hz)", linewidth=1.6)
    if show_events:
        for ev in ["StartHesitation", "Turn", "Walking"]:
            if ev in dfp.columns:
                plt.plot(dfp[time_col], dfp[ev], label=ev, alpha=0.5)
    plt.xlabel("Time (s)")
    plt.ylabel("Acceleration magnitude")
    plt.title(f"Patient {patient_id} – Acc Magnitude {title_suffix}")
    plt.legend(ncol=3)
    plt.grid(True)
    plt.tight_layout()
    plt.show()

# Add magnitude columns to dataframes
defog_df_bp = add_magnitude_cols(defog_df_bp)
tdcsfog_df_bp = add_magnitude_cols(tdcsfog_df_bp)
defog_df = add_magnitude_cols(defog_df)
tdcsfog_df = add_magnitude_cols(tdcsfog_df)

# Plot magnitude for one patient
plot_patient_mag(defog_df_bp, patient_id="4c3aa8ea6e", title_suffix="(raw vs band-pass)")


### BP Acceleration w/ FoG Events

In [ ]:
# Get patient IDs with FoG events
patient_id_defog = 'be9d33541d'
patient_id_tdcsfog = 'b670d1cddd' 

def _spans_from_mask(x, mask):
    """Convert a boolean mask to contiguous (start, end) intervals."""
    mask = np.asarray(mask, dtype=bool)
    if mask.size == 0:
        return []
    change = np.diff(mask.astype(int), prepend=0)
    starts = np.where(change == 1)[0]
    ends   = np.where(change == -1)[0] - 1
    if mask[0]:  starts = np.r_[0, starts]
    if mask[-1]: ends   = np.r_[ends, len(mask)-1]
    return [(x[s], x[e]) for s, e in zip(starts, ends)]

def plot_patient(ax, pl_df, pid, title):
    df = pl_df.filter(pl.col("patient_id") == pid).to_pandas()

    # Plot accelerations 
    for col in ["AccV_bp", "AccML_bp", "AccAP_bp"]:
        if col in df.columns:
            ax.plot(df["Time_in_sec"], df[col], label=col, alpha=0.8)

    if "StartHesitation" in df.columns:
        sh = pd.to_numeric(df["StartHesitation"], errors="coerce").fillna(0).to_numpy()
        mask = sh > 0
        spans = _spans_from_mask(df["Time_in_sec"].to_numpy(), mask)

        shaded_once = False
        for (start, end) in spans:
            ax.axvspan(start, end, alpha=0.25, label=(None if shaded_once else "StartHesitation"))
            shaded_once = True

    if "Turn" in df.columns:
        sh = pd.to_numeric(df["Turn"], errors="coerce").fillna(0).to_numpy()
        mask = sh > 0
        spans = _spans_from_mask(df["Time_in_sec"].to_numpy(), mask)

        shaded_once = False
        onset_once = False
        for (start, end) in spans:
            ax.axvspan(start, end, color="green", alpha=0.25, label=(None if shaded_once else "Turn"))
            shaded_once = True

    if "Walking" in df.columns:
        sh = pd.to_numeric(df["Walking"], errors="coerce").fillna(0).to_numpy()
        mask = sh > 0
        spans = _spans_from_mask(df["Time_in_sec"].to_numpy(), mask)

        shaded_once = False
        for (start, end) in spans:
            ax.axvspan(start, end, color="orange", alpha=0.25, label=(None if shaded_once else "Walking"))
            shaded_once = True
            
    ax.set_title(title)
    ax.set_xlabel("Time")
    ax.set_ylabel("Acceleration (g)")
    ax.grid(True)

# Create stacked subplots
fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(15, 10), sharex=False)

plot_patient(axes[0], defog_df_bp,   patient_id_defog,   f"DEFoG — {patient_id_defog}")
plot_patient(axes[1], tdcsfog_df_bp, patient_id_tdcsfog, f"TDCSFoG — {patient_id_tdcsfog}")

# Clean legends 
for ax in axes:
    h, l = ax.get_legend_handles_labels()
    seen, H, L = set(), [], []
    for hi, li in zip(h, l):
        if li and li not in seen:
            H.append(hi); L.append(li); seen.add(li)
    ax.legend(H, L, loc="upper right", ncol=3, fontsize=8)

fig.suptitle("BP Acceleration with FoG Highlights (DEFoG & tDCSFoG)", y=0.98)
plt.tight_layout()
plt.show() 

### Raw Acceleration v.s. BP Acceleration

In [ ]:
def plot_stacked_axes(raw_df: pl.DataFrame,
                      bp_df: pl.DataFrame,
                      patient_id: str,
                      dataset_name: str = "Dataset"):
   
    time_col = "Time_in_sec"
    axes_raw = ("AccV", "AccML", "AccAP")
    axes_bp  = tuple(f"{c}_bp" for c in axes_raw)
    colors = {"AccV": "tab:blue", "AccML": "tab:orange", "AccAP": "tab:green"}

    # Build pandas dataframe
    keep_raw = ["patient_id", time_col] + [c for c in axes_raw if c in raw_df.columns]
    keep_bp = ["patient_id", time_col] + [c for c in axes_bp if c in bp_df.columns]

    raw_pd = (
        raw_df.filter(pl.col("patient_id") == patient_id)
              .select(keep_raw)
              .to_pandas()
              .sort_values(time_col)
    )
    bp_pd = (
        bp_df.filter(pl.col("patient_id") == patient_id)
              .select(keep_bp)
              .to_pandas()
              .sort_values(time_col)
              .rename(columns={f"{c}_bp": c for c in axes_raw if f"{c}_bp" in bp_df.columns})
    )

    # Figure
    nrows = 2 * len(axes_raw)
    fig, axs = plt.subplots(nrows=nrows, ncols=1, figsize=(16, 4 * len(axes_raw)), sharex=True)
    axs = list(axs) if nrows > 1 else [axs]
    fig.suptitle(f"{dataset_name} {patient_id} — Raw vs Band-pass (stacked per axis)", y=0.995)

    row = 0
    for ax_name in axes_raw:
        if ax_name not in raw_pd.columns or ax_name not in bp_pd.columns:
            continue

        ymin = min(raw_pd[ax_name].min(), bp_pd[ax_name].min())
        ymax = max(raw_pd[ax_name].max(), bp_pd[ax_name].max())
        pad  = 0.05 * (ymax - ymin if ymax > ymin else 1.0)
        ylims = (ymin - pad, ymax + pad)

        # RAW
        axs[row].plot(raw_pd[time_col], raw_pd[ax_name],
                      color=colors.get(ax_name, "black"),
                      linewidth=1.0, label=f"{ax_name} (raw)")
        axs[row].set_ylabel(ax_name)
        axs[row].set_title(f"{ax_name} — Raw")
        axs[row].grid(True, alpha=0.3)
        axs[row].set_ylim(*ylims)
        axs[row].legend(loc="upper right", fontsize=8)
        row += 1

        # BAND-PASS
        axs[row].plot(bp_pd[time_col], bp_pd[ax_name],
                      color=colors.get(ax_name, "black"),
                      linewidth=1.0, label=f"{ax_name} (bp)")
        axs[row].set_ylabel(ax_name)
        axs[row].set_title(f"{ax_name} — Band-pass filtered")
        axs[row].grid(True, alpha=0.3)
        axs[row].set_ylim(*ylims)
        axs[row].legend(loc="upper right", fontsize=8)
        row += 1

    axs[-1].set_xlabel("Time (seconds)")
    plt.tight_layout(rect=[0, 0, 1, 0.98])
    plt.show()

#### Defog

In [ ]:
plot_stacked_axes(defog_df, defog_df_bp, patient_id="be9d33541d", dataset_name="DEFoG")

#### Tdcsfog

In [ ]:
plot_stacked_axes(tdcsfog_df, tdcsfog_df_bp, patient_id="b670d1cddd", dataset_name="TDCSFoG")

## Merge Defog & Tdcsfog 

In [ ]:
def prepare_defog(defog_df):
    # Drop rows where any FoG event is active but valid is false
    df = defog_df.filter(~(
            ((pl.col("StartHesitation") == 1) |
             (pl.col("Turn") == 1) |
             (pl.col("Walking") == 1))
            & (pl.col("Valid") == 0)
        )
    )

    # Drop valid and task columns
    df = df.drop(["Valid", "Task"])
    
    return df

defog_df_prep = prepare_defog(defog_df_bp)

In [ ]:
defog_df_prep.shape

In [ ]:
# Merge defog and tdcsfog
full_df = pl.concat([defog_df_prep, tdcsfog_df_bp], rechunk=True)

In [ ]:
full_df.shape

# Extract Time Domain Features 

In [ ]:
def extract_time_features_pl(
    df: pl.DataFrame,
    fs: float = 128.0,
    win_s: float = 2.0,
    hop_s: float = 0.5
) -> pl.DataFrame:
    """
    Extract time-domain features from accelerometer data.
    """

    acc_cols = ['AccV','AccML','AccAP']
    label_cols = ['StartHesitation', 'Turn', 'Walking'] 

    # --- window setup ---
    win = int(round(fs * win_s))
    hop = int(round(fs * hop_s))
   
    has_pid = "patient_id" in df.columns
    rows = []

    def _process_one_group(gdf: pl.DataFrame, pid_value=None):
        n = gdf.height
        if n < win:
            return  # not enough samples for one window

        X = gdf.select(acc_cols).to_numpy()
        labels = gdf.select(label_cols).to_numpy() if label_cols else None

        def corr_safe(a, b):
            if np.nanstd(a) == 0 or np.nanstd(b) == 0:
                return np.nan
            return np.corrcoef(a, b)[0, 1]

        win_id = 0
        for start in range(0, n - win + 1, hop):
            end = start + win
            W = X[start:end, :]
            f = {}

            # carry patient_id if present
            if pid_value is not None:
                f["patient_id"] = pid_value

            # per-axis stats
            for i, c in enumerate(acc_cols):
                w = W[:, i]
                mu = np.nanmean(w)
                sd = np.nanstd(w)
                f[f'{c}_mean'] = float(mu)
                f[f'{c}_std'] = float(sd)
                f[f'{c}_var'] = float(np.nanvar(w))
                f[f'{c}_min'] = float(np.nanmin(w))
                f[f'{c}_max'] = float(np.nanmax(w))
                f[f'{c}_median'] = float(np.nanmedian(w))
                q75, q25 = np.nanpercentile(w, [75, 25])
                f[f'{c}_iqr'] = float(q75 - q25)
                f[f'{c}_energy'] = float(np.nansum(w**2) / len(w))
                f[f'{c}_rms'] = float(np.sqrt(np.nanmean(w**2)))
                if sd > 0:
                    f[f'{c}_skew'] = float(((w - mu)**3).mean() / (sd**3))
                    f[f'{c}_kurt'] = float(((w - mu)**4).mean() / (sd**4) - 3)
                else:
                    f[f'{c}_skew'] = np.nan
                    f[f'{c}_kurt'] = np.nan

            # magnitude & SMA
            mag = np.sqrt(np.sum(W**2, axis=1))
            f['Acc_mag_mean'] = float(np.nanmean(mag))
            f['Acc_mag_std'] = float(np.nanstd(mag))
            f['Acc_sma'] = float(np.nansum(np.abs(W)) / len(W))

            # correlations
            f[f'corr_{acc_cols[0]}_{acc_cols[1]}'] = corr_safe(W[:,0], W[:,1])
            f[f'corr_{acc_cols[0]}_{acc_cols[2]}'] = corr_safe(W[:,0], W[:,2])
            f[f'corr_{acc_cols[1]}_{acc_cols[2]}'] = corr_safe(W[:,1], W[:,2])

            # labels
            if labels is not None:
                sub = labels[start:end, :]
                f['label_any'] = bool((sub > 0).any())
                for j, c in enumerate(label_cols):
                    f[f'label_{c}'] = bool((sub[:, j] > 0).any())

            # metadata
            f['win_id'] = win_id
            f['t_start_s'] = start / fs
            f['t_end_s'] = (end - 1) / fs

            rows.append(f)
            win_id += 1

    if has_pid:
        # group by patient and keep the id
        for (pid,), gdf in df.group_by(["patient_id"], maintain_order=True):
            _process_one_group(gdf, pid_value=pid)
    else:
        # single group (no patient_id)
        _process_one_group(df, pid_value=None)

    return pl.DataFrame(rows)


In [ ]:
FS = 128.0
WIN = 2.0
HOP = 0.5

full_feats = extract_time_features_pl(full_df, fs=FS, win_s=WIN, hop_s=HOP)

In [ ]:
full_feats.shape

In [ ]:
print(full_feats.columns)

# Extract Frequency Domain Features 
*  The Fourier Transform is a mathematical tool that takes a complex signal and breaks it down into its individual frequency components. It tells us what frequencies make up the signal and how strong each one is.
*  Time Domain features (mean, std, min, max, variance, median) shoes how a signal changes over time while a frequency domain feature (domiant frequency, spectral enegy, PSD) shows what frequencies make up the signal and how fast or periodic the motion is.
*  This helps us analyze how the signal's energy is distrubuted across different frequencies, revealing rhytmic motion patterns like walking, turning, or freezing of gait.
*  For PSD, we utilized Welch's method which smooths the FFT to give a more stable estimate of singal power over frequency. It does a great job detaching dominant movement frequencies (like steps per second)

Manual Frequency Features: 
* Directly linked to Parkinson's biomarkers like tremor, gait frequency, and freezing index
* Ability to tweak freqency bands, window sizes, or filtering parameters
  Quick to compute for large datasets 

In [ ]:
from scipy.fft import rfft, rfftfreq
from scipy.signal import welch

def extract_frequency_features_pl(
    df: pl.DataFrame, 
    fs: float = 128.0, 
    win_s: float = 2.0, 
    hop_s: float = 0.5,
    signal_cols: tuple = ('AccV', 'AccML', 'AccAP')
) -> pl.DataFrame:
    """
    Extract frequency-domain features from Polars DataFrame.
    Returns Polars DataFrame with PSD, FFT, band powers, and freezing index.
    """
    
    # Detect accelerometer columns
    cols = df.columns
    lower = {c.lower(): c for c in cols}
    candidates = [
        ['AccX','AccY','AccZ'],
        ['AccelX','AccelY','AccelZ'],
        ['acc_x','acc_y','acc_z'],
        ['AccV','AccML','AccAP'],
        ['accv','accml','accap'],
    ]
    acc_cols = None
    for trio in candidates:
        found = [lower.get(c.lower()) for c in trio]
        if all(found):
            acc_cols = found
            break
    if acc_cols is None:
        acc_cols = [c for c in cols if "acc" in c.lower()][:3]
    if len(acc_cols) < 3:
        raise KeyError(f"Could not find 3 accelerometer columns. Found: {acc_cols}")
    
    # Setup
    n = df.height
    win = int(round(fs * win_s))
    hop = int(round(fs * hop_s))
    
    # Extract numpy array for signal processing
    X = df.select(acc_cols).to_numpy()
    
    rows = []
    win_id = 0
    
    for start in range(0, n - win + 1, hop):
        end = start + win
        segment = X[start:end, :]
        fdict = {}
        
        # Frequency bins for FFT
        freqs = rfftfreq(win, d=1/fs)
        
        for i, col in enumerate(acc_cols):
            sig = segment[:, i]
            # Remove mean and handle NaNs
            sig = sig - np.nanmean(sig)
            sig = np.nan_to_num(sig)
            
            # FFT
            fft_vals = np.abs(rfft(sig))
            fft_power = fft_vals ** 2
            
            # PSD using Welch's method
            f_psd, psd = welch(sig, fs=fs, nperseg=min(win, len(sig)))
            
            # Basic FFT/PSD features
            fdict[f"{col}_fft_mean"] = float(np.mean(fft_power))
            fdict[f"{col}_fft_std"] = float(np.std(fft_power))
            fdict[f"{col}_fft_max"] = float(np.max(fft_power))
            fdict[f"{col}_psd_mean"] = float(np.mean(psd))
            fdict[f"{col}_psd_std"] = float(np.std(psd))
            fdict[f"{col}_psd_max"] = float(np.max(psd))
            fdict[f"{col}_dominant_freq"] = float(f_psd[np.argmax(psd)])
            
            # Band powers (FoG-relevant frequency bands)
            # 0-3 Hz: Normal gait/locomotion
            # 3-10 Hz: Tremor/FoG frequency
            # 10-30 Hz: High-frequency movement
            low_band = (f_psd >= 0) & (f_psd < 3)
            mid_band = (f_psd >= 3) & (f_psd < 10)
            high_band = (f_psd >= 10) & (f_psd < 30)
            
            low_power = np.trapz(psd[low_band], f_psd[low_band]) if low_band.any() else 0.0
            mid_power = np.trapz(psd[mid_band], f_psd[mid_band]) if mid_band.any() else 0.0
            high_power = np.trapz(psd[high_band], f_psd[high_band]) if high_band.any() else 0.0
            
            total_power = low_power + mid_power + high_power + 1e-9
            
            fdict[f"{col}_band_low"] = float(low_power)
            fdict[f"{col}_band_mid"] = float(mid_power)
            fdict[f"{col}_band_high"] = float(high_power)
            
            # Freezing Index (ratio of FoG band to total power)
            fdict[f"{col}_freezing_index"] = float(mid_power / total_power)
            
            # Spectral entropy (measure of signal complexity)
            psd_norm = psd / (np.sum(psd) + 1e-9)
            spectral_entropy = -np.sum(psd_norm * np.log2(psd_norm + 1e-9))
            fdict[f"{col}_spectral_entropy"] = float(spectral_entropy)
        
        # Metadata
        fdict['win_id'] = win_id
        fdict['t_start_s'] = start / fs
        fdict['t_end_s'] = (end - 1) / fs
        fdict['t_center_s'] = (fdict['t_start_s'] + fdict['t_end_s']) / 2.0
        
        rows.append(fdict)
        win_id += 1
    
    return pl.DataFrame(rows)

### Using tsfresh 
**Frequency Domain: FFT coefficients, spectral entropy, energy ratios, etc.**
* Time domain: mean, variance, autocorrelation, absolute energy, quantiles, etc.
* Complex statistics: number of peaks, linear trend slopes, precentage of reoccuring datapoints, and more.
  
**that allows our model to**
* Capture hidden micro-patterns in gait that's not visibilly obvious
* Use automated feature selection later (tsfresh.select_features())
* Complement your domain-driven features with new, data-driven ones - something revealing new insights 

In [ ]:
def extract_advanced_features_pl(
    df: pl.DataFrame,
    fs: float = 128.0,
    win_s: float = 2.0,
    hop_s: float = 0.5,
    signal_cols: tuple = ('AccV', 'AccML', 'AccAP')
) -> pl.DataFrame:
    """
    Extract advanced statistical features (tsfresh-like) from Polars DataFrame.
    Includes autocorrelation, complexity, peak detection, and more.
    """
    
    # Detect columns
    cols = df.columns
    lower = {c.lower(): c for c in cols}
    candidates = [
        ['AccX','AccY','AccZ'],
        ['AccelX','AccelY','AccelZ'],
        ['acc_x','acc_y','acc_z'],
        ['AccV','AccML','AccAP'],
        ['accv','accml','accap'],
    ]
    acc_cols = None
    for trio in candidates:
        found = [lower.get(c.lower()) for c in trio]
        if all(found):
            acc_cols = found
            break
    if acc_cols is None:
        acc_cols = [c for c in cols if "acc" in c.lower()][:3]
    
    # Setup
    n = df.height
    win = int(round(fs * win_s))
    hop = int(round(fs * hop_s))
    
    X = df.select(acc_cols).to_numpy()
    
    rows = []
    win_id = 0
    
    for start in range(0, n - win + 1, hop):
        end = start + win
        segment = X[start:end, :]
        fdict = {}
        
        for i, col in enumerate(acc_cols):
            sig = segment[:, i]
            sig = np.nan_to_num(sig)
            
            # Absolute energy
            fdict[f"{col}_abs_energy"] = float(np.sum(sig ** 2))
            
            # Absolute sum of changes
            fdict[f"{col}_abs_sum_changes"] = float(np.sum(np.abs(np.diff(sig))))
            
            # Autocorrelation (lag 1)
            if len(sig) > 1:
                sig_shifted = sig[1:]
                sig_orig = sig[:-1]
                if np.std(sig_orig) > 0 and np.std(sig_shifted) > 0:
                    autocorr = np.corrcoef(sig_orig, sig_shifted)[0, 1]
                else:
                    autocorr = 0.0
            else:
                autocorr = 0.0
            fdict[f"{col}_autocorr_lag1"] = float(autocorr)
            
            # Number of peaks (using simple threshold)
            mean_val = np.mean(sig)
            std_val = np.std(sig)
            threshold = mean_val + 0.5 * std_val
            peaks = 0
            for j in range(1, len(sig) - 1):
                if sig[j] > sig[j-1] and sig[j] > sig[j+1] and sig[j] > threshold:
                    peaks += 1
            fdict[f"{col}_num_peaks"] = peaks
            
            # Quantiles
            fdict[f"{col}_q25"] = float(np.percentile(sig, 25))
            fdict[f"{col}_q75"] = float(np.percentile(sig, 75))
            
            # Range
            fdict[f"{col}_range"] = float(np.max(sig) - np.min(sig))
            
            # Coefficient of variation
            cv = (np.std(sig) / (np.abs(np.mean(sig)) + 1e-9))
            fdict[f"{col}_coeff_variation"] = float(cv)
            
            # Zero crossing rate
            zero_crossings = np.sum(np.diff(np.sign(sig - np.mean(sig))) != 0)
            fdict[f"{col}_zero_crossing_rate"] = float(zero_crossings / len(sig))
            
            # Mean absolute deviation
            fdict[f"{col}_mean_abs_dev"] = float(np.mean(np.abs(sig - np.mean(sig))))
            
            # Linear trend slope (simple least squares)
            x_vals = np.arange(len(sig))
            if len(sig) > 1:
                slope = np.polyfit(x_vals, sig, 1)[0]
            else:
                slope = 0.0
            fdict[f"{col}_linear_trend_slope"] = float(slope)
        
        # Metadata
        fdict['win_id'] = win_id
        fdict['t_start_s'] = start / fs
        fdict['t_end_s'] = (end - 1) / fs
        fdict['t_center_s'] = (fdict['t_start_s'] + fdict['t_end_s']) / 2.0
        
        rows.append(fdict)
        win_id += 1
    
    return pl.DataFrame(rows)

In [ ]:
# Combines time + frequency + advanced features (all Polars-native!)
FS = 128.0
WIN = 2.0
HOP = 0.5

print("Extracting frequency-domain features...")
freq_feats = extract_frequency_features_pl(full_df, fs=FS, win_s=WIN, hop_s=HOP, signal_cols=('AccV','AccML','AccAP'))

print("Extracting advanced statistical features (tsfresh-like)...")
adv_feats = extract_advanced_features_pl(full_df, fs=FS, win_s=WIN, hop_s=HOP, signal_cols=('AccV','AccML','AccAP'))

# Merge all features on win_id (no need for time-based merge since windows are aligned!)
features_df = full_feats.join(
    freq_feats.drop(['t_start_s', 't_end_s', 't_center_s']),
    on='win_id',
    how='left'
).join(
    adv_feats.drop(['t_start_s', 't_end_s', 't_center_s']),
    on='win_id',
    how='left'
)

In [ ]:
print(f"Combined features_df shape: {features_df.shape}")
print(f"Total features: {len(features_df.columns)}")

print(f"\nColumns: {features_df.columns}")

# Feature Selection Using Mutual Information 
- The MI between two quantities is a measure of the extent to which knowledge of one quantity reduces uncertainty about the other.
- MI is similar to correlation, but with the advantage of being able to detect any kind of relationship, not just linear ones. Moreover, it is easy to use and interpret, computationally

In [ ]:
# Import mutual info selection libraries
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import mutual_info_classif
from sklearn.cluster import KMeans 
import random

### 1. Create 'Target' Multi-class Label
- Create a single, multi-class target column called target from three binary event columns (expected for mutual info)
- For each event column (e.g. "StartHesitation"):
  - Find all rows where that event column is 1
    - Set their new 'target' value to the name of the event (e.g. "StartHesitation")
  - Otherwise, if event column is 0
    - Set their new 'target' value to "Normal"

In [ ]:
# Build a multiclass target column
feats = features_df.with_columns(
    pl.when(pl.col("label_StartHesitation"))
      .then(pl.lit("StartHesitation"))
      .when(pl.col("label_Turn"))
      .then(pl.lit("Turn"))
      .when(pl.col("label_Walking"))
      .then(pl.lit("Walking"))
      .otherwise(pl.lit("Normal"))
      .alias("target")
)

### 2. Split into Train / Test 
- Split by continuous blocks (windows)
- df_train = Used to train the model (80%)
- df_test = Used to evaluate final performance (20%)

In [ ]:
# Assign block IDs based on win_id
BLOCK_SIZE = 1000  
feats = feats.with_columns((pl.col("win_id") // BLOCK_SIZE).alias("block_id"))

# Get unique block IDs and shuffle
blocks = feats.select("block_id").unique().to_series().to_list()
rng = random.Random(0)
rng.shuffle(blocks)

# 80/20 split for train/test
n = len(blocks)
n_train = int(0.8 * n)

b_train = set(blocks[:n_train])
b_test  = set(blocks[n_train:])

# Filter
df_train = feats.filter(pl.col("block_id").is_in(b_train))
df_test  = feats.filter(pl.col("block_id").is_in(b_test))

print("Train:", df_train.height, "Test:", df_test.height)

### 3. Fix Class Imbalance
- Fix class imbalance only on training dataset
- Use hybrid resampling 

In [ ]:
# Class distribution 
df_train['target'].value_counts(normalize=True)

In [ ]:
def hybrid_balance(df_train: pl.DataFrame, alpha: float = 2.0, seed: int = 0) -> pl.DataFrame:
    counts = df_train.group_by("target").agg(pl.len().alias("n"))
    classes = counts["target"].to_list()
    if "Normal" not in classes:
        return df_train

    minorities = counts.filter(pl.col("target") != "Normal")
    if minorities.is_empty():
        return df_train

    max_minority = int(minorities["n"].max())
    target_normal = int(alpha * max_minority)

    # Downsample Normal
    normal_blk = df_train.filter(pl.col("target") == "Normal")
    n_take = min(target_normal, normal_blk.height)
    if n_take < normal_blk.height:
        idx = np.random.RandomState(seed).choice(normal_blk.height, size=n_take, replace=False)
        normal_ds = normal_blk[idx.tolist()]  # slice by index list
    else:
        normal_ds = normal_blk

    # Upsample minorities
    parts = [normal_ds]
    turn_n = int(counts.filter(pl.col("target") == "Turn")["n"][0]) if "Turn" in classes else max_minority

    rng = np.random.RandomState(seed)
    for cls in ["Turn", "StartHesitation", "Walking"]:
        blk = df_train.filter(pl.col("target") == cls)
        if blk.is_empty():
            continue
        need = max(0, turn_n - blk.height)
        if need > 0:
            # random indices with replacement
            idx = rng.choice(blk.height, size=need, replace=True)
            blk_up = pl.concat([blk, blk[idx.tolist()]])
        else:
            blk_up = blk
        parts.append(blk_up)

    balanced = pl.concat(parts)

    # Manual shuffle by random permutation of row indices
    perm = np.random.RandomState(seed).permutation(balanced.height)
    balanced = balanced[perm.tolist()]

    return balanced


In [ ]:
# View class balance after hybrid resampling 
df_train_bal = hybrid_balance(df_train, alpha=2.0, seed=0)
df_train_bal['target'].value_counts(normalize=True)

### 4. Encode the target 
- Target must be encoded because it is categorical

In [ ]:
# Convert to pandas dataframe
df_train_pd = df_train_bal.to_pandas()

# Define features and target
target_col = "target"
feature_cols = [
    c for c in df_train_pd.columns
    if c not in [target_col, "t_start_s", "t_end_s", "win_id", "block_id", "patient_id"]  
       and not c.startswith("label_")
]

X = df_train_pd[feature_cols]
X = X.fillna(X.median(numeric_only=True))

y = df_train_pd[target_col]

In [ ]:
# Encode non-numerical target
le = LabelEncoder()
y_encode = le.fit_transform(y)

### 5. Compute & Visualize MI Scores

In [ ]:
# Compute mutual info
mi = mutual_info_classif(X, y_encode, random_state=0)
mi_scores = pd.Series(mi, index=X.columns, name="MI Scores").sort_values(ascending=False)
print(mi_scores.head(20))

In [ ]:
# Plot top 20 MI features 
TOP_K = 20
plt.figure(figsize=(10, 6))
topk = mi_scores.head(TOP_K).iloc[::-1] 
sns.barplot(x=topk.values, y=topk.index)
plt.title(f"Top {TOP_K} Mutual Information Features")
plt.xlabel("Mutual Information")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

In [ ]:
print(mi_scores.tail(20))

In [ ]:
# Plot bottom 20 MI features 
BOTTOM_K = 20
plt.figure(figsize=(10, 6))
bottomk = mi_scores.tail(BOTTOM_K).iloc[::-1]
sns.barplot(x=bottomk.values, y=bottomk.index)
plt.title(f"Lowest {BOTTOM_K} Mutual Information Features")
plt.xlabel("Mutual Information")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

# Model Selection
Comparing models for the following: 
- Random forest
- XG boost
- SVM (support vector machine)
- LSTM model
- Temporal CNN

In [ ]:
# Import all neccessary libraries 
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

## 1. Random Forest

In [ ]:
# Convert to pandas 
df_test_pd  = df_test.to_pandas()

# Features / target 
X_train = df_train_pd[feature_cols].copy()
y_train = df_train_pd[target_col].copy()

X_test = df_test_pd[feature_cols].copy()
y_test = df_test_pd[target_col].copy()

In [ ]:
print("Missing values in X_train:", X_train.isnull().sum().sum())
print("Missing values in y_train:", y_train.isnull().sum())

In [ ]:
df_train_pd['target'].value_counts(normalize=True)

In [ ]:
# Group by patient 
groups = df_train_pd["patient_id"]

### Random Forest evaluation:
- **Normal**: *Majority* predicted correctly
- **StartHesitation**: *None* predicted correctly
- **Turn**: *Less than half* predicted correctly
- **Walking**: *Very few* predicted correctly 

# 2. Support Vector Machine

In [ ]:
# Define pipeline for training linear SVM using SGD after imputing by median
# Define the pipeline with imputation and scaling
svm_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),  # Impute missing values with median
    ("scaler", StandardScaler()),  # Normalize features
    ("svm", SGDClassifier(loss='hinge',  # Linear SVM (hinge loss)
                           max_iter=1000,
                           tol=1e-3,
                           n_jobs=-1,
                           random_state=42))
])

In [ ]:
# Param grid for GridSearchCV
svm_param_grid = {
    'svm__alpha': [0.001, 0.01, 0.1, 1, 10],  # Regularization parameter (1/C)
    'svm__learning_rate': ['constant', 'optimal', 'invscaling'],
    'svm__eta0': [0.01, 0.1, 1],  # Initial learning rate for 'invscaling' and 'constant'
}

In [ ]:
# print("Missing values in X_train:", X_train.isnull().sum().sum())
# print("Missing values in y_train:", y_train.isnull().sum())

In [ ]:
# Group by patient 
groups = df_train_pd["patient_id"]

In [ ]:
# Setup StratifiedGroupKFold CV
svm_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

# Define GridSearchCV with cross-validation on the pipeline
grid_search = GridSearchCV(svm_pipe, svm_param_grid, cv=svm_cv, scoring='accuracy', verbose=1)

# Collect metrics for each fold
svm_accs, svm_f1s, svm_precisions, svm_roc_auc = [], [], [], []
for fold, (tr_idx, va_idx) in enumerate(svm_cv.split(X_train, y_train, groups=groups), start=1):
    X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
    y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[va_idx]

    svm_pipe.fit(X_tr, y_tr)
    y_hat = svm_pipe.predict(X_va)

    acc = accuracy_score(y_va, y_hat)
    f1 = f1_score(y_va, y_hat, average="binary" if len(np.unique(y_train))==2 else "macro")
    precision = precision_score(y_va, y_hat, average="binary" if len(np.unique(y_train))==2 else "macro")
    
    # Compute AUC only if there are two classes (binary classification)
    if len(np.unique(y_va)) == 2:
        fpr, tpr, _ = roc_curve(y_va, svm_pipe.predict_proba(X_va)[:, 1])
        roc_auc = auc(fpr, tpr)
    else:
        roc_auc = np.nan  # For multi-class, AUC calculation needs modification, handle as needed
    
    svm_accs.append(acc)
    svm_f1s.append(f1)
    svm_precisions.append(precision)
    svm_roc_auc.append(roc_auc)

    print(f"[Fold {fold}]  Acc: {acc:.4f}  F1: {f1:.4f}  Precision: {precision:.4f}  ROC AUC: {roc_auc:.4f}")

# After collecting metrics, compute the means and standard deviations
if len(svm_accs) > 0 and not all([np.isnan(x) for x in svm_accs]):
    acc_arr = np.array(svm_accs, dtype=float)
    acc_mean = np.nanmean(acc_arr)
    acc_std = np.nanstd(acc_arr, ddof=1) if np.sum(~np.isnan(acc_arr)) > 1 else 0.0
    print(f"\nCV mean Acc: {acc_mean:.4f} ± {acc_std:.4f}")

if len(svm_f1s) > 0 and not all([np.isnan(x) for x in svm_f1s]):
    f1_arr = np.array(svm_f1s, dtype=float)
    f1_mean = np.nanmean(f1_arr)
    f1_std = np.nanstd(f1_arr, ddof=1) if np.sum(~np.isnan(f1_arr)) > 1 else 0.0
    print(f"CV mean F1: {f1_mean:.4f} ± {f1_std:.4f}")

if len(svm_precisions) > 0:
    precision_arr = np.array(svm_precisions, dtype=float)
    precision_mean = np.nanmean(precision_arr)
    print(f"CV mean Precision: {precision_mean:.4f}")

if len(svm_roc_auc) > 0:
    auc_arr = np.array(svm_roc_auc, dtype=float)
    auc_mean = np.nanmean(auc_arr)
    print(f"CV mean ROC AUC: {auc_mean:.4f}")

In [ ]:
# Example to fit the model on training data
svm_pipe.fit(X_train, y_train)

# Example to make predictions and evaluate on the test data
test_accuracy = svm_pipe.score(X_test, y_test)
print(f"Test accuracy of the best model: {test_accuracy}")

In [ ]:
print("Grid Search CV Initiation")
# Use the grouped StratifiedGroupKFold if available
try:
    grid_search = GridSearchCV(svm_pipe, svm_param_grid, cv=svm_cv, scoring='accuracy', verbose=1)
except NameError:
    # fallback to regular cv=5 if svm_cv is not defined
    grid_search = GridSearchCV(svm_pipe, svm_param_grid, cv=5, scoring='accuracy', verbose=1)

print("Grid Search Fitting Starts Now")
# If groups is defined, pass it to fit for grouped CV
if 'groups' in globals():
    grid_search.fit(X_train, y_train, groups=groups)
else:
    grid_search.fit(X_train, y_train)

print(f"Best parameters from grid search: {grid_search.best_params_}")

In [1]:
# After grid search is complete, access the best pipeline (best_estimator_)
best_svm_pipe = grid_search.best_estimator_

y_pred = best_svm_pipe.predict(X_test)
y_prob = best_svm_pipe.predict_proba(X_test)[:, 1]  # Probabilities for ROC AUC

# Print results
print("\nTest Accuracy:", test_accuracy)
print("\nTest F1 Score:", test_f1)
print("\nTest Precision:", test_precision)
print("\nTest ROC AUC:", test_roc_auc)
print("\nClassification report:\n", classification_report(y_test, y_pred))

NameError: name 'grid_search' is not defined

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Normal', 'StartHesitation', 'Turn','Walking'], yticklabels=['Class 0', 'Class 1'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
report = classification_report(y_test, y_pred)
print("\nClassification Report:")
print(report)

In [ ]:
# Plot ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, color='b', label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.show()

In [ ]:
# Calibrated SVM (CalibratedClassifierCV)
# Wrap an SVM pipeline with CalibratedClassifierCV to improve probability estimates (may help ROC)
# import numpy as np
# import matplotlib.pyplot as plt
# from sklearn.svm import SVC
# from sklearn.calibration import CalibratedClassifierCV
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
# import seaborn as sns

# if 'X_train' not in globals() or 'y_train' not in globals():
#     raise NameError("`X_train` and `y_train` must be defined before running this cell.")
# if 'X_test' not in globals() or 'y_test' not in globals():
#     raise NameError("`X_test` and `y_test` must be defined before running this cell.")

# print('\n=== Improvement C: Calibrated SVM (CalibratedClassifierCV) ===')
# base_svc = Pipeline([
#     ('scaler', StandardScaler()),
#     ('svc', SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=42))
# ])

# calibrated = CalibratedClassifierCV(base_estimator=base_svc, cv=3)
# calibrated.fit(X_train, y_train)

# # Evaluate
# y_pred_cal = calibrated.predict(X_test)
# print('\nCalibrated SVM Classification Report:\n')
# print(classification_report(y_test, y_pred_cal))

# cm = confusion_matrix(y_test, y_pred_cal)
# plt.figure(figsize=(6,5))
# sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
# plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Calibrated SVM Confusion Matrix')
# plt.show()

# # ROC/AUC
# y_score_cal = calibrated.predict_proba(X_test)
# from sklearn.preprocessing import label_binarize
# classes = np.unique(y_test)
# y_test_bin = label_binarize(y_test, classes=classes)
# if y_score_cal.ndim == 1:
#     y_score_cal = np.vstack([1 - y_score_cal, y_score_cal]).T

# n_classes = classes.size
# fpr = dict(); tpr = dict(); roc_auc = dict()
# for i in range(n_classes):
#     fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_score_cal[:, i])
#     roc_auc[i] = auc(fpr[i], tpr[i])

# fpr['micro'], tpr['micro'], _ = roc_curve(y_test_bin.ravel(), y_score_cal.ravel())
# roc_auc['micro'] = auc(fpr['micro'], tpr['micro'])
# all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
# mean_tpr = np.zeros_like(all_fpr)
# for i in range(n_classes):
#     mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
# mean_tpr /= n_classes
# fpr['macro'] = all_fpr; tpr['macro'] = mean_tpr
# roc_auc['macro'] = auc(fpr['macro'], tpr['macro'])

# plt.figure(figsize=(8,6))
# plt.plot(fpr['micro'], tpr['micro'], label=f"micro-average (AUC = {roc_auc['micro']:.3f})", color='deeppink', linestyle=':', linewidth=3)
# plt.plot(fpr['macro'], tpr['macro'], label=f"macro-average (AUC = {roc_auc['macro']:.3f})", color='navy', linestyle=':', linewidth=3)
# colors = plt.cm.get_cmap('tab10')
# for i, cls in enumerate(classes):
#     plt.plot(fpr[i], tpr[i], color=colors(i % 10), lw=2, label=f"{cls} (AUC = {roc_auc[i]:.3f})")
# plt.plot([0,1],[0,1],'k--', lw=1)
# plt.xlim([0.0,1.0]); plt.ylim([0.0,1.05])
# plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
# plt.title('Calibrated SVM ROC (One-vs-Rest)')
# plt.legend(loc='lower right', fontsize='small'); plt.grid(True); plt.show()

# print('\nCalibrated SVM per-class AUCs:')
# for i, cls in enumerate(classes):
#     print(f" - {cls}: {roc_auc[i]:.4f}")
# print(f"micro-average AUC: {roc_auc['micro']:.4f}")
# print(f"macro-average AUC: {roc_auc['macro']:.4f}")

In [ ]:
# SVM with SMOTE (imblearn pipeline)
# Apply SMOTE to the training set in an imblearn pipeline, then scale and train SVM
# import sys
# import subprocess
# try:
#     from imblearn.over_sampling import SMOTE
#     from imblearn.pipeline import Pipeline as ImbPipeline
# except Exception:
#     subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'imbalanced-learn'])
#     from imblearn.over_sampling import SMOTE
#     from imblearn.pipeline import Pipeline as ImbPipeline

# from sklearn.preprocessing import StandardScaler
# from sklearn.svm import SVC
# from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
# import numpy as np
# import matplotlib.pyplot as plt
# import seaborn as sns

# if 'X_train' not in globals() or 'y_train' not in globals():
#     raise NameError("`X_train` and `y_train` must be defined before running this cell.")
# if 'X_test' not in globals() or 'y_test' not in globals():
#     raise NameError("`X_test` and `y_test` must be defined before running this cell.")

# print('\n=== Improvement B: SVM with SMOTE (resample training data) ===')
# smote_svc = ImbPipeline([
#     ('smote', SMOTE(random_state=42)),
#     ('scaler', StandardScaler()),
#     ('svc', SVC(kernel='rbf', probability=True, class_weight=None, random_state=42))
# ])

# smote_svc.fit(X_train, y_train)

# # Evaluate
# y_pred_sm = smote_svc.predict(X_test)
# print('\nSVM (SMOTE) Classification Report:\n')
# print(classification_report(y_test, y_pred_sm))

# cm = confusion_matrix(y_test, y_pred_sm)
# plt.figure(figsize=(6,5))
# sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
# plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('SVM (SMOTE) Confusion Matrix')
# plt.show()

# # ROC/AUC
# y_score_sm = smote_svc.predict_proba(X_test)
# from sklearn.preprocessing import label_binarize
# classes = np.unique(y_test)
# y_test_bin = label_binarize(y_test, classes=classes)
# if y_score_sm.ndim == 1:
#     y_score_sm = np.vstack([1 - y_score_sm, y_score_sm]).T

# n_classes = classes.size
# fpr = dict(); tpr = dict(); roc_auc = dict()
# for i in range(n_classes):
#     fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_score_sm[:, i])
#     roc_auc[i] = auc(fpr[i], tpr[i])

# fpr['micro'], tpr['micro'], _ = roc_curve(y_test_bin.ravel(), y_score_sm.ravel())
# roc_auc['micro'] = auc(fpr['micro'], tpr['micro'])
# all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
# mean_tpr = np.zeros_like(all_fpr)
# for i in range(n_classes):
#     mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
# mean_tpr /= n_classes
# fpr['macro'] = all_fpr; tpr['macro'] = mean_tpr
# roc_auc['macro'] = auc(fpr['macro'], tpr['macro'])

# plt.figure(figsize=(8,6))
# plt.plot(fpr['micro'], tpr['micro'], label=f"micro-average (AUC = {roc_auc['micro']:.3f})", color='deeppink', linestyle=':', linewidth=3)
# plt.plot(fpr['macro'], tpr['macro'], label=f"macro-average (AUC = {roc_auc['macro']:.3f})", color='navy', linestyle=':', linewidth=3)
# colors = plt.cm.get_cmap('tab10')
# for i, cls in enumerate(classes):
#     plt.plot(fpr[i], tpr[i], color=colors(i % 10), lw=2, label=f"{cls} (AUC = {roc_auc[i]:.3f})")
# plt.plot([0,1],[0,1],'k--', lw=1)
# plt.xlim([0.0,1.0]); plt.ylim([0.0,1.05])
# plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
# plt.title('SVM (SMOTE) ROC (One-vs-Rest)')
# plt.legend(loc='lower right', fontsize='small'); plt.grid(True); plt.show()

# print('\nSVM (SMOTE) per-class AUCs:')
# for i, cls in enumerate(classes):
#     print(f" - {cls}: {roc_auc[i]:.4f}")
# print(f"micro-average AUC: {roc_auc['micro']:.4f}")
# print(f"macro-average AUC: {roc_auc['macro']:.4f}")

In [ ]:
# Improvement A — SVM with class_weight='balanced'
# Re-train a simple SVM pipeline that compensates for class imbalance and evaluate it
# import numpy as np
# import matplotlib.pyplot as plt
# from sklearn.pipeline import Pipeline
# from sklearn.preprocessing import StandardScaler
# from sklearn.svm import SVC
# from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
# import seaborn as sns

# if 'X_train' not in globals() or 'y_train' not in globals():
#     raise NameError("`X_train` and `y_train` must be defined before running this cell.")
# if 'X_test' not in globals() or 'y_test' not in globals():
#     raise NameError("`X_test` and `y_test` must be defined before running this cell.")

# print('\n=== Improvement A: Re-training SVM with class_weight="balanced" ===')
# balanced_svc = Pipeline([
#     ('scaler', StandardScaler()),
#     ('svc', SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=42))
# ])

# balanced_svc.fit(X_train, y_train)

# # Evaluate
# y_pred_bal = balanced_svc.predict(X_test)
# print('\nBalanced SVM Classification Report:\n')
# print(classification_report(y_test, y_pred_bal))

# Confusion matrix plot
# cm = confusion_matrix(y_test, y_pred_bal)
# plt.figure(figsize=(6,5))
# sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
# plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Balanced SVM Confusion Matrix')
# plt.show()

# # ROC / multiclass AUC
# from sklearn.preprocessing import label_binarize
# classes = np.unique(y_test)
# y_test_bin = label_binarize(y_test, classes=classes)
# y_score_bal = balanced_svc.predict_proba(X_test)

# if y_score_bal.ndim == 1:
#     y_score_bal = np.vstack([1 - y_score_bal, y_score_bal]).T

# n_classes = classes.size
# fpr = dict(); tpr = dict(); roc_auc = dict()
# for i in range(n_classes):
#     fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_score_bal[:, i])
#     roc_auc[i] = auc(fpr[i], tpr[i])

# fpr['micro'], tpr['micro'], _ = roc_curve(y_test_bin.ravel(), y_score_bal.ravel())
# roc_auc['micro'] = auc(fpr['micro'], tpr['micro'])
# all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
# mean_tpr = np.zeros_like(all_fpr)
# for i in range(n_classes):
#     mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
# mean_tpr /= n_classes
# fpr['macro'] = all_fpr; tpr['macro'] = mean_tpr
# roc_auc['macro'] = auc(fpr['macro'], tpr['macro'])

# plt.figure(figsize=(8,6))
# plt.plot(fpr['micro'], tpr['micro'], label=f"micro-average (AUC = {roc_auc['micro']:.3f})", color='deeppink', linestyle=':', linewidth=3)
# plt.plot(fpr['macro'], tpr['macro'], label=f"macro-average (AUC = {roc_auc['macro']:.3f})", color='navy', linestyle=':', linewidth=3)
# colors = plt.cm.get_cmap('tab10')
# for i, cls in enumerate(classes):
#     plt.plot(fpr[i], tpr[i], color=colors(i % 10), lw=2, label=f"{cls} (AUC = {roc_auc[i]:.3f})")
# plt.plot([0,1],[0,1],'k--', lw=1)
# plt.xlim([0.0,1.0]); plt.ylim([0.0,1.05])
# plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
# plt.title('Balanced SVM ROC (One-vs-Rest)')
# plt.legend(loc='lower right', fontsize='small'); plt.grid(True); plt.show()

# print('\nBalanced SVM per-class AUCs:')
# for i, cls in enumerate(classes):
#     print(f" - {cls}: {roc_auc[i]:.4f}")
# print(f"micro-average AUC: {roc_auc['micro']:.4f}")
# print(f"macro-average AUC: {roc_auc['macro']:.4f}")
